# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dharshinianand19/FlyRank-Starters/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Contract:** One row represents one pseudonymized content item. The starter dataset is a
content-level snapshot with trailing-90-day performance metrics, not a daily time-series table.

The dataset contains 30,000 rows and 44 columns across 32 pseudonymized clients. `content_id`
identifies one content item, while `client_id` is context for grouping and grouped train/test
splits. Because the starter CSV does not expose a report date, the observed metrics are treated
as a single trailing-90-day snapshot window.

In [9]:

import pandas as pd

url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Rows:", len(df))
print("Columns:", df.shape[1])
print("Unique content items:", df["content_id"].nunique())
print("Unique clients:", df["client_id"].nunique())

duplicate_content_ids = (
    df.groupby("content_id")
      .size()
      .loc[lambda s: s > 1]
)

print("Duplicate content_id groups observed:", len(duplicate_content_ids))
print(duplicate_content_ids)

Rows: 30000
Columns: 44
Unique content items: 30000
Unique clients: 32
Duplicate content_id groups observed: 0
Series([], dtype: int64)


**Field contract:**

- **Features:** `content_type`, `word_count`, and availability flags such as `has_word_count`.
  These describe the content before predicting future performance decline.
- **Label / proxy:** `is_declining_label`, `trend_direction`, and `trend_pct`. The label is
  derived from `trend_direction`, which is computed from `trend_pct`; therefore none of these
  fields may be used as features.
- **Context:** `content_id` and `client_id`. These are pseudonymous identifiers used only for
  identification, grouping, joining, and grouped train/test splits.
- **Excluded:** Any private, product-decision, or future-looking fields identified in the data
  dictionary. They are excluded because they could leak outcome information or violate privacy.

Rate fields are stored as ×100 percentages: `ctr = 0.76` means 0.76%, not 76%. This applies to
`ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, and `trend_pct`. Also,
`avg_position = 0` means “no data,” not rank zero.

In [10]:
label_proxy_fields = ["is_declining_label", "trend_direction", "trend_pct"]
context_fields = ["content_id", "client_id"]
rate_fields = [
    "ctr", "engagement_rate", "scroll_rate",
    "ai_traffic_pct", "trend_pct"
]

available_label_fields = [c for c in label_proxy_fields if c in df.columns]
available_context_fields = [c for c in context_fields if c in df.columns]
available_rate_fields = [c for c in rate_fields if c in df.columns]

print("Label/proxy fields observed:", available_label_fields)
print("Context fields observed:", available_context_fields)
print("Rate fields observed:", available_rate_fields)

if available_rate_fields:
    display(df[available_rate_fields].describe())

if "avg_position" in df.columns:
    print("Rows with avg_position == 0, meaning no position data:",
          int((df["avg_position"] == 0).sum()))

Label/proxy fields observed: ['trend_direction', 'trend_pct']
Context fields observed: ['content_id', 'client_id']
Rate fields observed: ['ctr', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'trend_pct']


,ctr,engagement_rate,scroll_rate,ai_traffic_pct,trend_pct
count,30000.000000,30000.000000,29875.000000,30000.000000,26612.000000
mean,0.510733,2.534520,18.212921,0.768196,-4.785969
std,3.279162,8.310096,29.472768,7.429454,473.861780
min,0.000000,0.000000,0.000000,0.000000,-100.000000
25%,0.000000,0.000000,0.000000,0.000000,-62.600000
50%,0.070000,0.000000,5.000000,0.000000,-33.500000
75%,0.290000,1.350000,23.530000,0.000000,0.000000
max,100.000000,100.000000,300.000000,300.000000,44900.000000


Rows with avg_position == 0, meaning no position data: 1205


**Verification:** The checks below verify the stated grain, expected counts, patterned
missingness, and the interpretation of zero-valued position data.

- The grain check should return zero duplicate `content_id` groups.
- The row count should be 30,000, with 30,000 unique content items and 32 clients.
- Missingness is checked by `content_type` because missing keyword and word-count data may
  follow content type.
- `avg_position = 0` is checked separately because it represents missing data, not rank zero.

In [11]:
print("GRAIN CHECK")
print("Duplicate content_id groups:", len(duplicate_content_ids))

print("\nCOUNT CHECK")
print("Total rows:", len(df))
print("Unique content_id values:", df["content_id"].nunique())
print("Unique client_id values:", df["client_id"].nunique())

print("\nMISSINGNESS CHECK")
missing_overall = df.isna().mean().sort_values(ascending=False)
display(missing_overall.head(15))

check_cols = [c for c in ["keyword_data", "word_count"] if c in df.columns]

if check_cols:
    missing_by_content_type = (
        df.groupby("content_type")[check_cols]
          .apply(lambda group: group.isna().mean())
    )
    print("\nMissingness by content_type:")
    display(missing_by_content_type)

    for col in check_cols:
        df[f"has_{col}"] = df[col].notna().astype(int)

    print("\nAvailability flags created:")
    print(df[[f"has_{col}" for col in check_cols]].mean())

print("\nZERO-POSITION CHECK")
if "avg_position" in df.columns:
    print("Rows where avg_position == 0:",
          int((df["avg_position"] == 0).sum()))

GRAIN CHECK
Duplicate content_id groups: 0

COUNT CHECK
Total rows: 30000
Unique content_id values: 30000
Unique client_id values: 32

MISSINGNESS CHECK


,0
provider_used,0.714600
word_count,0.256633
char_count,0.256633
word_count_tier,0.256633
char_count_tier,0.256633
model_used,0.191100
trend_pct,0.112933
competition_level,0.087000
search_volume,0.082267
cpc,0.082267



Missingness by content_type:


,word_count
content_type,
comparison article,0.000000
feedly article,0.000000
keyword article,0.282979



Availability flags created:
has_word_count    0.743367
dtype: float64

ZERO-POSITION CHECK
Rows where avg_position == 0: 1205


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*
**Data limits:** This dataset can support directional, decision-support analysis of which
content items show observed decline signals. It cannot prove that a content change caused a
performance change, because the data is observational and does not include a controlled
experiment.

The starter snapshot provides trailing-90-day metrics but does not expose a report date, so it
cannot support time-based train/test splits by itself. Client histories may be unbalanced, and
early warehouse rows may be GSC-only before GA4 data becomes available; GA4 zeros before
`ga4_data_start` should be treated as unavailable rather than as zero engagement.

The label is derived from `trend_direction`, which is computed from `trend_pct`. Therefore,
`trend_pct` and `trend_direction` cannot be used as features. Missing values also follow
`content_type`, so a blanket `fillna(0)` would create a misleading category signal; explicit
availability flags are used instead.

In [12]:
print("Data-limit checks")

print("Rows:", len(df))
print("Unique clients:", df["client_id"].nunique())

if "ga4_data_available" in df.columns:
    print("GA4 availability distribution:")
    print(df["ga4_data_available"].value_counts(dropna=False))
else:
    print("GA4 availability flag is not present in the starter CSV.")

if "trend_pct" in df.columns and "trend_direction" in df.columns:
    print("\nLabel-proxy fields present and excluded from features:")
    print(["trend_pct", "trend_direction", "is_declining_label"])

print("\nThese limits mean results are observational and decision-support only,")
print("not causal evidence.")

Data-limit checks
Rows: 30000
Unique clients: 32
GA4 availability flag is not present in the starter CSV.

Label-proxy fields present and excluded from features:
['trend_pct', 'trend_direction', 'is_declining_label']

These limits mean results are observational and decision-support only,
not causal evidence.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.